# FEV / GIFT REST API Evaluation (SAP AI Core)

This notebook reproduces FEV and GIFT-Eval benchmark results by calling the SAP-RPT-TS
deployment on **SAP AI Core** (`POST $DEPLOYMENT_URL/predict`) and feeding the responses
into the same official evaluators used by the local runs.

**Authentication.** Provide an AI Core service key (`credential-type: x509`) and set
`KEY_FILE` and `DEPLOYMENT_URL` in the config cell below. Authentication (mTLS token
exchange) is handled internally by `ai_api_client_sdk`.

**Environment.** Set up FEV and GIFT-Eval per their official instructions. Validated with
`fev==0.11.0.dev0`, `salesforce-gift-eval==0.0.0a0`, `gluonts==0.17.0`, `datasets==4.8.5`.
If either benchmark is updated later (task definitions, windowing, metrics, or dataset
versions), results may no longer apply and must be re-baselined.

In [ ]:
import os
from pathlib import Path

# --- SAP AI Core deployment (x509 service key over mTLS) ---
KEY_FILE       = "<path-to-your-service-key>.json"   # AI Core service-key JSON
DEPLOYMENT_URL = "<your-AI-Core-deployment-URL>"     # deployment URL (without /predict)
RESOURCE_GROUP = "default"

# FEV task definitions from the public fev-bench benchmark
FEV_TASKS_YAML = "https://raw.githubusercontent.com/autogluon/fev/refs/heads/main/benchmarks/fev_bench/tasks.yaml"

# Example: run one FEV task.
# To run all 100 FEV tasks set FEV_TASK_NAMES = None.
FEV_TASK_NAMES = ["proenfo_gfc12"]

# GIFT: point GIFT_EVAL at your local data directory before running.
# Download instructions: https://github.com/SalesforceAIResearch/gift-eval
# os.environ["GIFT_EVAL"] = "/path/to/gift_dataset"

# Example: run one GIFT task.
# To run all 97 GIFT tasks set GIFT_TASKS = None.
GIFT_TASKS = [("ett1/H", "short")]

OUTPUT_DIR = Path("./api_eval_output")
OUTPUT_DIR.mkdir(exist_ok=True)

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
BATCH_SIZE  = 128  # number of series per POST request (HTTP chunk size)
MAX_WORKERS = 4    # parallel POST threads

print(f'FEV tasks: {FEV_TASK_NAMES or "all"}')
print(f'GIFT tasks: {GIFT_TASKS or "all"}')

## REST API client

In [ ]:
import json, math, time
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from ai_api_client_sdk.ai_api_v2_client import AIAPIV2Client


def _to_jsonable(obj):
    """Recursively convert numpy/pandas objects to JSON-serialisable types. NaN -> null."""
    if isinstance(obj, float):         return None if math.isnan(obj) else obj
    if isinstance(obj, np.floating):   return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.ndarray):    return _to_jsonable(obj.tolist())
    if isinstance(obj, (list, tuple)): return [_to_jsonable(x) for x in obj]
    if isinstance(obj, dict):          return {k: _to_jsonable(v) for k, v in obj.items()}
    tp = type(obj)
    if tp.__module__.startswith("pandas") and tp.__name__ == "Timestamp":
        return obj.isoformat(sep=" ", timespec="seconds")[:19]
    return obj


# --- SAP AI Core client (x509 / mTLS handled by SDK) ---
_key = json.loads(Path(KEY_FILE).read_text())
_client = AIAPIV2Client(
    base_url=_key["serviceurls"]["AI_API_URL"] + "/v2",
    auth_url=_key["certurl"].rstrip("/") + "/oauth/token",
    client_id=_key["clientid"],
    cert_str=_key["certificate"],
    key_str=_key["key"],
    resource_group=RESOURCE_GROUP,
    num_request_retries=3,
    read_timeout=600,
)
# Path suffix relative to base_url, e.g. /inference/deployments/<id>/predict
_PREDICT_PATH = DEPLOYMENT_URL.split("/v2", 1)[-1].rstrip("/") + "/predict"


def api_predict(horizon, data, options=None, quantile_levels=None):
    """POST /predict. Token refresh and retries are handled by the SDK.

    The SDK's default body camelization is disabled (convert_body_to_camel_case=False)
    so that option keys reach the server in snake_case as the API expects.
    """
    payload = {"horizon": int(horizon), "data": _to_jsonable(data)}
    opts = dict(options or {})
    if quantile_levels:
        opts["quantile_levels"] = quantile_levels
        opts.setdefault("output_quantile", True)
    if opts:
        payload["options"] = opts
    return _client.rest_client.post(path=_PREDICT_PATH, body=payload,
                                    convert_body_to_camel_case=False)


def extract_forecasts(response, *, n_series, horizon, n_variates_per_series, quantile_levels):
    """Normalise API response to ndarray (n_series, max_variates, horizon, n_quantiles).

    Prefers selected_quantile_result (C, Q, H) when present; falls back to result field.
    """
    Q, max_v = len(quantile_levels), max(n_variates_per_series)
    out = np.zeros((n_series, max_v, horizon, Q))
    if response.get("selected_quantile_result") is not None:
        for i, s in enumerate(response["selected_quantile_result"]):
            arr = np.asarray(s)
            if arr.ndim == 2: arr = arr[np.newaxis]
            arr = arr.transpose(0, 2, 1)          # (C,Q,H) -> (C,H,Q)
            C = n_variates_per_series[i]
            out[i, :C] = arr[:C]
        return out
    for i, s in enumerate(response["result"]):
        arr = np.asarray(s, dtype=np.float64)
        C = n_variates_per_series[i]
        if   arr.ndim == 1:                              arr = np.broadcast_to(arr[None, :, None], (1, horizon, Q))
        elif arr.ndim == 2 and arr.shape == (horizon, Q): arr = arr[None]
        elif arr.ndim == 2:                              arr = np.broadcast_to(arr[:, :, None], (C, horizon, Q))
        elif arr.ndim == 3 and arr.shape[1] == Q:        arr = arr.transpose(0, 2, 1)
        out[i, :C] = arr
    return out

print("AI Core API client ready")

## FEV evaluation

For each task and each sliding window, call the API, then pass the results to
`task.evaluation_summary()` to compute SQL / MASE / WAPE / WQL.

In [ ]:
import datasets
import fev
import pandas as pd


def _fev_window_to_inputs(window):
    """Convert an EvaluationWindow to a list of API input dicts."""
    past_df, future_df, _ = fev.convert_input_data(window, adapter="pandas")
    target_cols = list(window.target_columns)
    past_dyn    = list(window.past_dynamic_columns)
    known_dyn   = list(window.known_dynamic_columns)
    future_groups = (
        {str(sid): f.sort_values("timestamp", kind="stable")
         for sid, f in future_df.groupby("id", sort=False)}
        if future_df is not None else {}
    )
    inputs = []
    for _, hist in past_df.groupby("id", sort=False):
        hist = hist.sort_values("timestamp", kind="stable")
        sid  = str(hist["id"].iloc[0])
        item = {"target": hist[target_cols].to_numpy(dtype=float).T}
        if past_dyn or known_dyn:
            item["past_covariates"] = {c: hist[c].to_numpy() for c in [*past_dyn, *known_dyn]}
        if known_dyn:
            fut = future_groups[sid]
            item["future_covariates"] = {c: fut[c].to_numpy() for c in known_dyn}
        inputs.append(item)
    return inputs, target_cols


def _reorder_covariates(inp):
    """Sort past_covariates to match Chronos2 channel order: past-only first, known-future last."""
    past, future = inp.get("past_covariates"), inp.get("future_covariates")
    if not past: return inp
    fk = set(future) if future else set()
    sk = sorted(past)
    r = dict(inp)
    r["past_covariates"] = {k: past[k] for k in sk if k not in fk}
    r["past_covariates"].update({k: past[k] for k in sk if k in fk})
    if future: r["future_covariates"] = {k: future[k] for k in sk if k in fk}
    return r


def predict_fev_via_api(task):
    """Run API inference for all sliding windows of a fev.Task.

    Returns predictions_per_window (list[DatasetDict]) ready for task.evaluation_summary().
    HTTP chunks are aligned to cross-learning group boundaries so grouping matches local runs.
    """
    task.load_full_dataset(num_proc=1)
    predictions_per_window = []

    for w_idx, window in enumerate(task.iter_windows()):
        inputs, target_cols = _fev_window_to_inputs(window)
        inputs = [_reorder_covariates(i) for i in inputs]
        n_variates = [np.atleast_2d(i["target"]).shape[0] for i in inputs]
        H = window.horizon

        # Align HTTP chunk size to cross-learning group boundaries
        rows_per = np.atleast_2d(inputs[0]["target"]).shape[0] + len(inputs[0].get("past_covariates") or {})
        group_n  = max(1, math.ceil(20 / rows_per))
        chunk    = max(group_n, (BATCH_SIZE // group_n) * group_n)
        b_starts = list(range(0, len(inputs), chunk))

        def _post(b, _inp=inputs, _nv=n_variates, _H=H):
            bi, bn = _inp[b:b+chunk], _nv[b:b+chunk]
            body = api_predict(_H, bi,
                               options={"cross_learning": True,
                                        "batch_size": 20,
                                        "enhanced_forecast": True},
                               quantile_levels=QUANTILE_LEVELS)
            return b, extract_forecasts(body, n_series=len(bi), horizon=_H,
                                        n_variates_per_series=bn, quantile_levels=QUANTILE_LEVELS)

        all_fc = np.zeros((len(inputs), max(n_variates), H, len(QUANTILE_LEVELS)))
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            for b, fc in pool.map(_post, b_starts):
                all_fc[b:b+fc.shape[0], :fc.shape[1]] = fc

        # Build the DatasetDict format expected by task.evaluation_summary()
        mid = QUANTILE_LEVELS.index(0.5)
        preds = datasets.DatasetDict({
            v: datasets.Dataset.from_dict(
                {"predictions": all_fc[:, vi, :, mid],
                 **{str(q): all_fc[:, vi, :, qi] for qi, q in enumerate(QUANTILE_LEVELS)}}
            )
            for vi, v in enumerate(target_cols)
        })
        preds.set_format("numpy")
        predictions_per_window.append(preds)
        print(f"  window {w_idx+1}/{task.num_windows}")

    return predictions_per_window


# Load benchmark
benchmark = fev.Benchmark.from_yaml(str(FEV_TASKS_YAML))
if FEV_TASK_NAMES is None:
    tasks_to_run = benchmark.tasks
else:
    tasks_to_run = [t for t in benchmark.tasks
                    if (getattr(t, "task_name", None) or getattr(t, "dataset_config", None))
                    in FEV_TASK_NAMES]

print(f"FEV tasks to run: {len(tasks_to_run)}")

In [ ]:
fev_summaries = []
for task in tasks_to_run:
    task_name = getattr(task, "task_name", None) or getattr(task, "dataset_config", None)
    print(f"\n[FEV] {task_name}  horizon={task.horizon}  windows={task.num_windows}")
    t0 = time.time()
    try:
        preds = predict_fev_via_api(task)
        summary = task.evaluation_summary(
            preds,
            model_name=f"remote-api::{DEPLOYMENT_URL}",
            inference_time_s=time.time() - t0,
        )
        fev_summaries.append(summary)
        print(f"  SQL={summary.get('SQL'):.4f}  MASE={summary.get('MASE'):.4f}  "
              f"WAPE={summary.get('WAPE'):.4f}  WQL={summary.get('WQL'):.4f}")
    except Exception as e:
        print(f"  ERROR: {e}")

fev_df = pd.DataFrame(fev_summaries)
fev_csv = OUTPUT_DIR / "fev_api_results.csv"
fev_df.to_csv(fev_csv, index=False)
print(f"\nResults written to: {fev_csv}")
_cols = [c for c in ["dataset_config", "SQL", "MASE", "WAPE", "WQL"] if c in fev_df.columns]
fev_df[_cols] if _cols else fev_df


## GIFT evaluation

For each dataset/term, call the API to obtain quantile forecasts, then pass them to
`gluonts.model.evaluate_forecasts()` to compute the metrics.

In [ ]:
import itertools
from gluonts.ev.metrics import MAE, MASE, ND, MeanWeightedSumQuantileLoss
from gluonts.model import evaluate_forecasts
from gluonts.model.forecast import QuantileForecast
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset as GiftDataset

GIFT_METRICS = [
    MASE(),
    MAE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=QUANTILE_LEVELS),
]

# Full GIFT task list (short term for all datasets; medium/long only for high-frequency ones)
_SHORT    = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
_MED_LONG = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"

def _all_gift_tasks():
    med_long_set = set(_MED_LONG.split())
    tasks = []
    for ds in sorted(set(_SHORT.split())):
        for term in ["short", "medium", "long"]:
            if term in ("medium", "long") and ds not in med_long_set:
                continue
            tasks.append((ds, term))
    return tasks

gift_tasks = GIFT_TASKS if GIFT_TASKS is not None else _all_gift_tasks()
print(f"GIFT tasks to run: {len(gift_tasks)}")

In [ ]:
_PRETTY = {"saugeenday": "saugeen", "temperature_rain_with_missing": "temperature_rain",
           "kdd_cup_2018_with_missing": "kdd_cup_2018", "car_parts_with_missing": "car_parts"}


def predict_gift_via_api(dataset: GiftDataset) -> list:
    """Run API inference for all sliding windows of a GiftDataset.

    Returns list[QuantileForecast] ready for gluonts.model.evaluate_forecasts().
    """
    pred_len    = dataset.prediction_length
    n_windows   = dataset.test_data.windows
    inputs_list = list(dataset.test_data.input)
    forecast_windows = []

    for w_idx in range(n_windows):
        window_inputs = list(itertools.islice(inputs_list, w_idx, None, n_windows))
        b_starts = list(range(0, len(window_inputs), BATCH_SIZE))

        def _post(b, _wi=window_inputs):
            batch = _wi[b:b+BATCH_SIZE]
            api_batch, nvars = [], []
            for item in batch:
                t = np.asarray(item["target"], dtype=np.float64)
                t2 = t[np.newaxis] if t.ndim == 1 else t
                nvars.append(t2.shape[0])
                api_batch.append({"target": t2.tolist()})
            body = api_predict(pred_len, api_batch,
                               options={"cross_learning": True,
                                        "batch_size": 100,
                                        "enhanced_forecast": True},
                               quantile_levels=QUANTILE_LEVELS)
            fc = extract_forecasts(body, n_series=len(api_batch), horizon=pred_len,
                                   n_variates_per_series=nvars, quantile_levels=QUANTILE_LEVELS)
            return b, fc

        batch_results = {}
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            for b, fc in pool.map(_post, b_starts):
                batch_results[b] = fc

        win_forecasts = []
        for b in b_starts:
            fc = batch_results[b]
            for i, ts in enumerate(window_inputs[b:b+BATCH_SIZE]):
                t = np.asarray(ts["target"], dtype=np.float64)
                # fc[i, 0, :, :] shape: (pred_len, Q) -> transpose to (Q, pred_len)
                arrays = fc[i, 0, :, :].T
                start_date = ts["start"] + len(t.ravel() if t.ndim == 1 else t[0])
                win_forecasts.append(QuantileForecast(
                    forecast_arrays=arrays,
                    forecast_keys=[str(q) for q in QUANTILE_LEVELS],
                    start_date=start_date,
                ))
        forecast_windows.append(win_forecasts)
        print(f"  window {w_idx+1}/{n_windows}")

    # Interleave windows back to series-major order
    return [item for items in zip(*forecast_windows) for item in items]


def _ds_config_key(ds_name, freq_str, term):
    base = _PRETTY.get(ds_name.split("/")[0].lower(), ds_name.split("/")[0].lower())
    return f"{base}/{freq_str}/{term}"

print("GIFT helpers ready")

In [ ]:
gift_rows = []
for ds_name, term in gift_tasks:
    print(f"\n[GIFT] {ds_name}/{term}")
    try:
        dataset = GiftDataset(name=ds_name, term=term, to_univariate=True)
        print(f"  freq={dataset.freq}  pred_len={dataset.prediction_length}  windows={dataset.test_data.windows}")

        forecasts = predict_gift_via_api(dataset)

        result = evaluate_forecasts(
            forecasts,
            test_data=dataset.test_data,
            metrics=GIFT_METRICS,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=get_seasonality(dataset.freq),
        ).reset_index(drop=True).to_dict(orient="records")[0]

        freq_seg = str(dataset.freq).split("-")[0]
        ds_key = _ds_config_key(ds_name, freq_seg, term)
        print(f"  MASE={result.get('MASE[0.5]'):.4f}  CRPS={result.get('mean_weighted_sum_quantile_loss'):.4f}")
        gift_rows.append({"dataset": ds_key, **{f"eval_metrics/{k}": v for k, v in result.items()}})
    except Exception as e:
        print(f"  ERROR: {e}")
        gift_rows.append({"dataset": f"{ds_name}/{term}", "error": repr(e)})

gift_df = pd.DataFrame(gift_rows)
gift_csv = OUTPUT_DIR / "gift_api_results.csv"
gift_df.to_csv(gift_csv, index=False)
print(f"\nResults written to: {gift_csv}")
gift_df